<!-- civilpy-title-block -->
<div style="border-top:6px solid #0E3F75; border-bottom:2px solid #C12637; padding:0.6em 0 0.6em 0; margin-bottom:1.2em;">
<span style="font-size:1.5em; font-weight:bold; color:#0E3F75;">BIM Cost Estimate from a Rhino Model</span><br>
<span style="color:#54585A;">Ohio Department of Transportation &middot; CivilPy Notebook Series</span>
</div>

# BIM Cost Estimate from a Rhino Model

**The estimate comes from the `.3dm` file alone — no Rhino session, no
spreadsheet, no re-takeoff.**

Every object the BrIM emit layer draws carries a typed attribute record as
Rhino user text:

- `bim.*` — component type, id, and (where the part is a standard detail)
  the ODOT SCD it came from;
- `mat.*` — material spec/grade/class;
- `pay.*` — the ODOT pay item **and its quantity contribution**.

Because the attributes live on the objects, the saved Rhino document is the
BIM record: reading the file back with `rhino3dm` (a pip install, no Rhino
license) regenerates the quantity takeoff and prices it. This notebook does
that against `AI Generated Bridge.3dm` — the same LOD 350 model from the
Rhino demo.

In [1]:
from pathlib import Path

import pandas as pd

from civilpy.structural.rhino_bim import (
    read_bim_estimate, read_bim_quantities, read_bim_tags)

MODEL = Path("Rhino Components/Rhino/AI Generated Bridge.3dm")

record = read_bim_tags(MODEL)
bridge = record["bridge"]
print(f"{MODEL.name}: {len(record['components']):,} tagged components\n")
print("bridge-wide record (the bim.type = 'bridge' marker):")
for k, v in sorted(bridge.items()):
    print(f"  {k:<26}{v}")

AI Generated Bridge.3dm: 15,209 tagged components

bridge-wide record (the bim.type = 'bridge' marker):
  bim.composite             true
  bim.cross_slope_pct       2
  bim.girder_count          5
  bim.girder_label          W40X211
  bim.girder_spacing_ft     9
  bim.id                    BRIDGE
  bim.overhang_ft           2
  bim.railing               SBR-1-20
  bim.scd_year              2020
  bim.skew_deg              0
  bim.spans_ft              120,150,150,150,120
  bim.type                  bridge
  bim.units                 ft
  gdr.deck_fc               4.5
  gdr.deck_t                7.5
  gdr.deck_weff             108


## What one component's record looks like

Each part is self-describing. A girder knows its shape, steel grade, and
pay-item weight; a bar knows its size, coating, bend shape, and weight —
enough that the estimate (and, through the preserved `gdr.*` tags, the
analysis model) can be regenerated by anyone who has the file.

In [2]:
def show(tags, title):
    print(title)
    for k, v in sorted(tags.items()):
        print(f"  {k:<26}{v}")
    print()

comps = record["components"]
show(next(c for c in comps if c["bim.type"] == "girder"),
     "a girder:")
show(next(c for c in comps if c["bim.type"] == "rebar"),
     "a reinforcing bar:")

a girder:
  bim.id                    G1
  bim.type                  girder
  girder.shape              W40X211
  mat.grade                 50W
  mat.spec                  ASTM A709
  mat.treatment             none
  mat.type                  weathering steel
  pay.category              513
  pay.desc                  Structural steel members, Level 1
  pay.item                  513E10220
  pay.level                 1
  pay.qty                   145590
  pay.unit                  lb

a reinforcing bar:
  bim.id                    BAR-LONG-TOP-1
  bim.type                  rebar
  mat.spec                  reinforcing steel
  pay.category              509
  pay.desc                  Epoxy coated reinforcing steel [CONFIRM]
  pay.item                  509E00200
  pay.level                 1
  pay.qty                   460.697
  pay.unit                  lb
  rebar.bend                straight
  rebar.coating             epoxy
  rebar.dia_in              0.5
  rebar.length_ft           68

## Quantity takeoff — a walk over the file

`read_bim_quantities` groups every visible component by its `pay.item` and
totals the `pay.qty` contributions. Objects on hidden layers are excluded —
this model keeps a superseded copy of the superstructure under a hidden
`Legacy` layer for reference, and the takeoff counts exactly what the
viewport shows.

In [3]:
q = read_bim_quantities(MODEL)
pd.DataFrame([
    {"item": item, "description": r["desc"], "unit": r["unit"],
     "quantity": r["qty"], "objects": r["objects"]}
    for item, r in q.items()
]).set_index("item")

,description,unit,quantity,objects
item,,,,
507E10000,"Steel piles HP, furnished and driven [CONFIRM]",ft,320.00,8
509E00200,Epoxy coated reinforcing steel [CONFIRM],lb,254936.52,9852
509E00300,GFRP deformed bars [CONFIRM],lb,0.00,24
511E12100,"Class QC2 concrete, superstructure (deck) [CON...",cy,757.86,6
511E40000,"Class QC1 concrete, substructure [CONFIRM]",cy,613.62,62
512E10000,"Concrete, parapet/railing [CONFIRM]",cy,208.70,2
513E10220,"Structural steel members, Level 1",lb,733577.34,35
513E20000,Shear connectors (welded studs),ea,5175.00,5175
516E10000,Elastomeric bearing [CONFIRM],ea,30.00,30


## Pricing the takeoff

`read_bim_estimate` runs the same rollup through a unit-price book. The
built-in `DEFAULT_UNIT_PRICES` are **planning-level placeholders** (round
numbers in the range of recent ODOT bid tabs) — pass `prices={...}` to
override any item with district estimate numbers.

In [4]:
est = read_bim_estimate(MODEL)

df = pd.DataFrame([
    {"item": item, "description": r["desc"], "unit": r["unit"],
     "quantity": r["qty"], "unit price ($)": r["unit_price"],
     "cost ($)": r["cost"]}
    for item, r in est.rows.items()
]).set_index("item")
display(df.style.format({"quantity": "{:,.1f}", "unit price ($)": "{:,.2f}",
                         "cost ($)": "{:,.0f}"}))

length_ft = sum(float(s) for s in bridge["bim.spans_ft"].split(","))
width_ft = ((int(bridge["bim.girder_count"]) - 1)
            * float(bridge["bim.girder_spacing_ft"])
            + 2 * float(bridge["bim.overhang_ft"]))
deck_sf = length_ft * width_ft
print(f"structural items total:  ${est.total:>12,.0f}")
print(f"deck area: {length_ft:.0f} ft x {width_ft:.0f} ft = {deck_sf:,.0f} sf"
      f"  ->  ${est.total / deck_sf:,.0f}/sf")

,description,unit,quantity,unit price ($),cost ($)
item,,,,,
507E10000,"Steel piles HP, furnished and driven [CONFIRM]",ft,320.0,75.00,"24,000"
509E00200,Epoxy coated reinforcing steel [CONFIRM],lb,"254,936.5",1.60,"407,898"
509E00300,GFRP deformed bars [CONFIRM],lb,0.0,3.00,0
511E12100,"Class QC2 concrete, superstructure (deck) [CONFIRM]",cy,757.9,950.00,"719,967"
511E40000,"Class QC1 concrete, substructure [CONFIRM]",cy,613.6,850.00,"521,577"
512E10000,"Concrete, parapet/railing [CONFIRM]",cy,208.7,"1,200.00","250,440"
513E10220,"Structural steel members, Level 1",lb,"733,577.3",2.25,"1,650,549"
513E20000,Shear connectors (welded studs),ea,"5,175.0",10.00,"51,750"
516E10000,Elastomeric bearing [CONFIRM],ea,30.0,"2,000.00","60,000"


structural items total:  $   3,686,181
deck area: 690 ft x 40 ft = 27,600 sf  ->  $134/sf


Overriding the book with project numbers is one argument — here, steel at
a fabricated-curved-girder price and bearings from a supplier quote:

In [5]:
est_proj = read_bim_estimate(
    MODEL, prices={"513E10220": 2.85, "516E10000": 3200.0})
print(f"default book:   ${est.total:>12,.0f}")
print(f"project prices: ${est_proj.total:>12,.0f}")

default book:   $   3,686,181
project prices: $   4,162,328


## Any tagged model prices the same way

The same one-liner works on every file the emit layer produced — here the
substructure type gallery from the LOD 350 demo:

In [6]:
gallery = Path("Rhino Components/Rhino/Substructure Type Gallery.3dm")
est_g = read_bim_estimate(gallery)
print(f"{gallery.name}")
print(est_g)

Substructure Type Gallery.3dm
item                   qty unit     unit $        cost $  description
507E10000            490.0 ft        75.00        36,750  Steel piles HP, furnished and driven [CONFIRM]
509E00200         81,665.9 lb         1.60       130,666  Epoxy coated reinforcing steel [CONFIRM]
509E00300              0.0 lb         3.00             0  GFRP deformed bars [CONFIRM]
511E12100            288.8 cy       950.00       274,398  Class QC2 concrete, superstructure (deck) [CONFIRM]
511E40000             73.2 cy       850.00        62,246  Class QC1 concrete, substructure [CONFIRM]
512E10000             87.7 cy     1,200.00       105,264  Concrete, parapet/railing [CONFIRM]
513E10220        176,250.9 lb         2.25       396,565  Structural steel members, Level 1
513E20000          1,740.0 ea        10.00        17,400  Shear connectors (welded studs)
516E10000             12.0 ea     2,000.00        24,000  Elastomeric bearing [CONFIRM]
total                             

---
## Where this leaves us

- **The record is the file.** The `.3dm` alone — opened with the free
  `rhino3dm` package — carries the component inventory, materials,
  pay items, quantities, and now a priced estimate.
- Quantities honor layer visibility, so superseded geometry parked under a
  hidden `Legacy` layer never leaks into the takeoff.
- Prices are a swappable book: planning defaults out of the box,
  district/bid-tab numbers with one argument.

Related walkthroughs: **Steel Girder Bridge Walkthrough** (§5–6 draw this
model and roll the quantities from the live emit), **MIDAS Bridge Model
Walkthrough** (the analysis side of the same record).